# Notebook 4 — Multiple Linear Regression & Evaluation

Notebook này sử dụng trực tiếp `prepared_invoice_dataset_cleaned.csv` được tạo từ Notebook 2.

Mục tiêu:
- xây dựng mô hình hồi quy tuyến tính đa biến
- one-hot encoding cho biến phân loại
- train/test split
- đánh giá bằng MAE, RMSE, R²
- diễn giải hệ số quan trọng

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [ ]:
BASE_DIR = Path.cwd()
prepared_path = BASE_DIR / 'prepared_invoice_dataset_cleaned.csv'

if not prepared_path.exists():
    raise FileNotFoundError('Bạn hãy chạy Notebook 2 trước để tạo prepared_invoice_dataset_cleaned.csv')

df = pd.read_csv(prepared_path)
df['invoice_date'] = pd.to_datetime(df['invoice_date'], errors='coerce')

print('Modeling dataset shape:', df.shape)
display(df.head())

In [ ]:
# Chọn biến mục tiêu và biến đầu vào
target = 'total_amount'
feature_candidates = [
    'service_category',
    'payment_method_group',
    'invoice_status',
    'month',
    'quarter',
    'year',
    'item_count',
    'total_quantity',
    'avg_unit_price',
    'max_item_subtotal',
    'is_cancelled',
]

features = [col for col in feature_candidates if col in df.columns]
X = df[features].copy()
y = df[target].copy()

categorical_features = X.select_dtypes(include=['object']).columns.tolist()
numeric_features = [col for col in features if col not in categorical_features]

print('Features used:', features)
print('Categorical:', categorical_features)
print('Numeric:', numeric_features)

# Lưu ý: không đưa trực tiếp subtotal_before_tax hay total_tax vào mô hình
# để tránh rò rỉ thông tin quá gần với total_amount.


In [ ]:
# Chia train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f'MAE  = {mae:,.0f}')
print(f'RMSE = {rmse:,.0f}')
print(f'R²   = {r2:.4f}')


In [ ]:
# So sánh giá trị thực tế và dự báo
results = pd.DataFrame({
    'actual_total_amount': y_test.values,
    'predicted_total_amount': y_pred
})
results['error'] = results['actual_total_amount'] - results['predicted_total_amount']
display(results.head(10))

plt.figure(figsize=(7, 5))
plt.scatter(results['actual_total_amount'], results['predicted_total_amount'])
plt.title('Actual vs Predicted')
plt.xlabel('Actual Total Amount')
plt.ylabel('Predicted Total Amount')
plt.tight_layout()
plt.show()

In [ ]:
# Diễn giải hệ số mô hình tuyến tính
preprocessor_fitted = model.named_steps['preprocessor']
regressor = model.named_steps['regressor']
feature_names = preprocessor_fitted.get_feature_names_out()
coefficients = pd.DataFrame({
    'feature': feature_names,
    'coefficient': regressor.coef_
}).sort_values('coefficient', ascending=False)

display(coefficients.head(15))
display(coefficients.tail(15))